# bansho - Feature Importance Scoring

This notebook demonstrates `megumi.bansho.score_features` using **synthetic datasets with a known ground truth**. Because we generate the data ourselves, we know exactly which features carry signal, so we can verify that bansho recovers them correctly.

| Example | Generator | Task |
|---------|-----------|------|
| 1 | `make_classification` | Binary classification |
| 2 | `make_regression` | Regression |

**How bansho works:**

1. Two synthetic `RANDOM_1` / `RANDOM_2` columns (standard normal) are injected into the feature matrix.
2. A random forest is fitted on the extended matrix.
3. Mean absolute SHAP values are computed via `TreeExplainer`.
4. Each original feature is labelled relative to the random baselines:

| Label | Condition |
|-------|-----------|
| `predictive` | mean\|SHAP\| > max(RANDOM_1, RANDOM_2) - genuine signal |
| `marginal`   | min < mean\|SHAP\| ≤ max - weak signal |
| `noise`      | mean\|SHAP\| ≤ min(RANDOM_1, RANDOM_2) - no detectable signal |

---

> **Why we don't use randomly drawn noise features here.**
>
> `RANDOM_1` and `RANDOM_2`, the internal baselines bansho injects, are i.i.d. standard normal draws. A noise column created with `make_classification`'s unused feature slots, or simply appended from `np.random.randn`, is drawn from *exactly the same distribution*. Whether such a column ranks above or below the baselines in mean absolute SHAP is pure chance: with enough noise columns, some will look `predictive` on any given seed, not because they carry signal but because of random fluctuations in the SHAP estimates. The demo would be misleading.
>
> Instead, we deliberately construct non-predictive features in ways that have nothing to do with fresh random draws:
> - **Permuted features**: values copied from an informative column and row shuffled; the marginal distribution is identical to the original, but the relationship with the target is completely destroyed.
> - **Constant features**: a single value repeated for every row; zero variance by construction.
> - **Semi constant features**: 97% of rows share the same value, the remaining 3% carry a tiny Gaussian jitter; practically zero variance.
>
> A model cannot extract any signal from these columns regardless of the random seed, making them a far more honest test of bansho's ability to distinguish signal from noise.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from megumi.bansho import score_features

/Users/eligoze/miniforge3/envs/megumi-dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## Example 1: Binary Classification

We generate a dataset with **15 features** split into five groups:

| Group | Count | Description |
|-------|-------|-------------|
| `informative_*` | 5 | Directly predictive of the target |
| `redundant_*`   | 3 | Linear combinations of the informative features, correlated with the target but carry no independent signal |
| `permuted_*`    | 3 | Copies of informative features with rows randomly shuffled, same marginal distribution, zero relationship with the target |
| `constant_*`    | 2 | A single repeated value for all rows, zero variance by construction |
| `near_constant_*` | 2 | 97 % of rows share the same value, 3 % carry a tiny Gaussian jitter, near-zero variance |

We expect bansho to classify all `informative_*` and most `redundant_*` features as **predictive**. Permuted, constant, and near constant features should land predominantly in **noise**, any that slip into **marginal** or even **predictive** do so by chance (spurious random-forest splits), not because they carry real signal.

In [ ]:
N_SAMPLES = 2_000
N_INFORMATIVE_CLF = 5
N_REDUNDANT = 3
N_PERMUTED_CLF = 3
N_NEAR_CONSTANT_CLF = 2

rng_clf = np.random.default_rng(42)

# Only request informative + redundant columns from sklearn (no sklearn noise slots)
X_clf, y_clf = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_INFORMATIVE_CLF + N_REDUNDANT,
    n_informative=N_INFORMATIVE_CLF,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=2,
    shuffle=False,
    random_state=42,
)

df_clf = pd.DataFrame(
    X_clf,
    columns=(
        [f"informative_{i+1}" for i in range(N_INFORMATIVE_CLF)]
        + [f"redundant_{i+1}" for i in range(N_REDUNDANT)]
    ),
)
df_clf["target"] = y_clf

# Permuted features: copy an informative column and shuffle its rows.
# The marginal distribution is unchanged; only the alignment with the target is destroyed.
for i in range(N_PERMUTED_CLF):
    src = f"informative_{i+1}"
    df_clf[f"permuted_{i+1}"] = rng_clf.permutation(df_clf[src].values)

# Constant features: one repeated value for every row, zero variance by construction.
df_clf["constant_1"] = 0.0
df_clf["constant_2"] = 5.0

# Near-constant features: 97% identical, 3% tiny Gaussian jitter.
for i in range(N_NEAR_CONSTANT_CLF):
    vals = np.zeros(N_SAMPLES)
    mask = rng_clf.random(N_SAMPLES) < 0.03
    vals[mask] = rng_clf.standard_normal(mask.sum()) * 0.1
    df_clf[f"near_constant_{i+1}"] = vals

feature_names_clf = [c for c in df_clf.columns if c != "target"]

print(f"Shape: {df_clf.shape}")
print(f"Target distribution:\n{df_clf['target'].value_counts().to_string()}")
df_clf.head(3)

Shape: (2000, 16)
Target distribution:
target
0    1000
1    1000


,informative_1,informative_2,informative_3,informative_4,informative_5,redundant_1,redundant_2,redundant_3,target,permuted_1,permuted_2,permuted_3,constant_1,constant_2,near_constant_1,near_constant_2
0,0.642928,-0.621520,2.663515,-0.599783,0.335670,0.886828,0.140331,-1.679709,0,2.450405,1.723084,-1.414179,0.0,5.0,0.0,0.0
1,-0.070846,2.901385,-0.832169,-1.501497,-0.212647,-0.616619,-3.085321,2.030037,0,0.500356,-2.119524,-1.629876,0.0,5.0,0.0,0.0
2,0.876182,2.158993,1.566156,-1.121890,-0.151822,-0.080111,-1.739280,0.330297,0,0.046662,2.446657,-0.578897,0.0,5.0,0.0,0.0


In [3]:
df_clf_train, df_clf_test = train_test_split(df_clf, test_size=0.2, random_state=42)

result_clf = score_features(
    df_clf_train,
    features=feature_names_clf,
    target="target",
    df_val=df_clf_test,
    random_state=42,
)
result_clf

,feature,predictive_power
0,informative_4,predictive
1,informative_2,predictive
2,informative_1,predictive
3,redundant_2,predictive
4,informative_5,predictive
5,informative_3,predictive
6,redundant_1,predictive
7,redundant_3,predictive
8,permuted_1,predictive
9,permuted_3,noise


---

## Example 2: Regression

We generate a dataset with **12 features** split into four groups:

| Group | Count | Description |
|-------|-------|-------------|
| `informative_*` | 5 | Linearly predictive of the target |
| `permuted_*`    | 3 | Copies of informative features with rows randomly shuffled, same marginal distribution, zero relationship with the target |
| `constant_*`    | 2 | A single repeated value for all rows (no variance by construction) |
| `near_constant_*` | 2 | 97% of rows share the same value, 3% carry a tiny Gaussian jitter, near-zero variance |

A moderate noise level is added to make the task realistic. We expect bansho to classify all `informative_*` as **predictive**. Permuted, constant, and near-constant features should land predominantly in **noise**, any that drift into **marginal** or **predictive** do so by chance, not because of genuine signal.

In [4]:
N_INFORMATIVE_REG = 5
N_PERMUTED_REG = 3
N_NEAR_CONSTANT_REG = 2

rng_reg = np.random.default_rng(0)

# Only request informative columns from sklearn (no sklearn noise slots)
X_reg, y_reg = make_regression(
    n_samples=N_SAMPLES,
    n_features=N_INFORMATIVE_REG,
    n_informative=N_INFORMATIVE_REG,
    noise=30.0,
    shuffle=False,
    random_state=42,
)

df_reg = pd.DataFrame(
    X_reg,
    columns=[f"informative_{i+1}" for i in range(N_INFORMATIVE_REG)],
)
df_reg["target"] = y_reg

# Permuted features: copy an informative column and shuffle its rows.
for i in range(N_PERMUTED_REG):
    src = f"informative_{i+1}"
    df_reg[f"permuted_{i+1}"] = rng_reg.permutation(df_reg[src].values)

# Constant features: one repeated value for every row — zero variance by construction.
df_reg["constant_1"] = 0.0
df_reg["constant_2"] = 5.0

# Near-constant features: 97 % identical, 3 % tiny Gaussian jitter.
for i in range(N_NEAR_CONSTANT_REG):
    vals = np.zeros(N_SAMPLES)
    mask = rng_reg.random(N_SAMPLES) < 0.03
    vals[mask] = rng_reg.standard_normal(mask.sum()) * 0.1
    df_reg[f"near_constant_{i+1}"] = vals

feature_names_reg = [c for c in df_reg.columns if c != "target"]

print(f"Shape: {df_reg.shape}")
print(f"Target stats:\n{df_reg['target'].describe().to_string()}")
df_reg.head(3)

Shape: (2000, 13)
Target stats:
count    2000.000000
mean        0.467802
std        74.044045
min      -305.280468
25%       -47.107135
50%         0.649646
75%        50.417610
max       279.660318


,informative_1,informative_2,informative_3,informative_4,informative_5,target,permuted_1,permuted_2,permuted_3,constant_1,constant_2,near_constant_1,near_constant_2
0,0.496714,-0.138264,0.647689,1.523030,-0.234153,15.450865,-1.277437,0.606723,0.218718,0.0,5.0,0.0,0.0
1,-0.234137,1.579213,0.767435,-0.469474,0.542560,29.240155,0.964156,0.846840,-0.507717,0.0,5.0,0.0,0.0
2,-0.463418,-0.465730,0.241962,-1.913280,-1.724918,-161.247558,0.946808,-0.796064,0.960932,0.0,5.0,0.0,0.0


In [5]:
df_reg_train, df_reg_test = train_test_split(df_reg, test_size=0.2, random_state=42)

result_reg = score_features(
    df_reg_train,
    features=feature_names_reg,
    target="target",
    df_val=df_reg_test,
    random_state=42,
)
result_reg

,feature,predictive_power
0,informative_5,predictive
1,informative_1,predictive
2,informative_2,predictive
3,informative_4,predictive
4,informative_3,predictive
5,permuted_2,marginal
6,permuted_3,marginal
7,permuted_1,noise
8,near_constant_2,noise
9,near_constant_1,noise


---

## Summary

A side-by-side view of both runs: how many features in each true group ended up in each bansho tier.

In [6]:
def build_summary(result: pd.DataFrame, dataset: str, task: str) -> pd.DataFrame:
    counts = (
        result["predictive_power"]
        .value_counts()
        .reindex(["predictive", "marginal", "noise"], fill_value=0)
        .to_frame(name="count")
        .T
    )
    counts.insert(0, "task", task)
    counts.insert(0, "dataset", dataset)
    counts.index = [""]
    return counts


summary = pd.concat(
    [
        build_summary(result_clf, "classification", "binary"),
        build_summary(result_reg, "regression", "continuous"),
    ],
    ignore_index=True,
)

summary

predictive_power,dataset,task,predictive,marginal,noise
0,classification,binary,9,0,6
1,regression,continuous,5,2,5
